[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openai/openai-cookbook/blob/main/articles/gpt-oss/run-colab.ipynb)

# 在免费的 Google Colab 中运行 OpenAI gpt-oss 20B

OpenAI 发布了 `gpt-oss` [120B](https://hf.co/openai/gpt-oss-120b) 和 [20B](https://hf.co/openai/gpt-oss-20b) 两个模型。两者均采用 Apache 2.0 许可证。

具体来说，`gpt-oss-20b` 专为更低延迟以及本地或专门化的使用场景而设计（210 亿参数，其中 36 亿为激活参数）。

由于这些模型是以原生 MXFP4 量化方式训练的，因此即使在 Google Colab 这样资源受限的环境中，也能轻松运行 20B 模型。

作者：[Pedro](https://huggingface.co/pcuenq) 和 [VB](https://huggingface.co/reach-vb)

## 设置环境

由于 transformers 对 mxfp4 的支持还处于非常前沿的阶段，我们需要较新版本的 PyTorch 和 CUDA，才能安装 `mxfp4` triton 内核。

我们还需要从源码安装 transformers，并卸载 `torchvision` 和 `torchaudio` 以消除依赖冲突。

In [ ]:
!pip install -q --upgrade torch

In [ ]:
!pip install -q transformers triton==3.4 kernels

In [ ]:
!pip uninstall -q torchvision torchaudio -y

安装完上述软件包后，请重启你的 Colab 运行时会话。

## 在 Google Colab 中从 Hugging Face 加载模型

我们从这里加载模型：[openai/gpt-oss-20b](https://hf.co/openai/gpt-oss-20b)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="cuda",
)

## 设置消息/对话

你可以提供一个可选的系统提示词，也可以直接提供输入。

In [ ]:
messages = [
    {"role": "system", "content": "Always respond in riddles"},
    {"role": "user", "content": "What is the weather like in Madrid?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))

## 指定推理强度

只需将其作为额外参数传给 `apply_chat_template()` 即可。支持的取值为 `"low"`、`"medium"`（默认）或 `"high"`。

In [ ]:
messages = [
    {"role": "system", "content": "Always respond in riddles"},
    {"role": "user", "content": "Explain why the meaning of life is 42"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="high",
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))

## 尝试其他提示词和想法！

查看我们的博客文章，获取更多想法：[https://hf.co/blog/welcome-openai-gpt-oss](https://hf.co/blog/welcome-openai-gpt-oss)